In [1]:
import rasterio
import numpy as np

# -------- USER INPUTS --------
r_bathtub  = r"D:\Phd Research\Final_Raster\Bathtub_depth_100yr_surge_SLR.tif"   # OR your bathtub path
r_process  = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"  # process-based
bin_width  = 0.3
x_max      = 12.0
# -----------------------------

def depths_and_pixel_area(path, clip_max=None):
    """Return 1D depths >0 (<=clip_max if set) and pixel area (m^2) if CRS is projected."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        nd  = src.nodata
        valid = np.isfinite(arr) if nd is None else ((arr != nd) & np.isfinite(arr))
        vals  = arr[valid]
        vals  = vals[vals > 0]
        if clip_max is not None:
            vals = vals[vals <= clip_max]

        pix_area = None
        if src.crs and not src.crs.is_geographic:
            # pixel width*height (m^2) — works for projected CRS in meters
            pix_area = abs(src.transform.a) * abs(src.transform.e)
        else:
            raise RuntimeError(
                f"{path} appears to be in a geographic CRS (degrees). "
                "Reproject to a metric CRS (e.g., UTM) before computing areas."
            )
    return vals, pix_area

# Common bins
bins = np.arange(0, x_max + bin_width, bin_width)

# Read + bin
vals_bath, pix_bath = depths_and_pixel_area(r_bathtub, clip_max=x_max)
vals_proc, pix_proc = depths_and_pixel_area(r_process,  clip_max=x_max)

counts_bath, edges = np.histogram(vals_bath, bins=bins)
counts_proc, _     = np.histogram(vals_proc, bins=bins)

# Convert to area per bin (km^2)
area_bath_bins = counts_bath * (pix_bath / 1e6)   # km^2
area_proc_bins = counts_proc * (pix_proc / 1e6)   # km^2

# Overlap in histogram space (common part per bin)
overlap_bins = np.minimum(area_bath_bins, area_proc_bins)

# Unique (non-overlapping) area per bin
unique_bath_bins = area_bath_bins - overlap_bins
unique_proc_bins = area_proc_bins - overlap_bins

# Totals
total_bath_area_km2 = area_bath_bins.sum()
total_proc_area_km2 = area_proc_bins.sum()
overlap_area_km2    = overlap_bins.sum()

unique_bath_total_km2 = unique_bath_bins.sum()
unique_proc_total_km2 = unique_proc_bins.sum()

print("\n=== Histogram-based area summary (km²) ===")
print(f"Bathtub (orange) total area:         {total_bath_area_km2:,.2f}")
print(f"Process-based (blue) total area:      {total_proc_area_km2:,.2f}")
print(f"Overlapping area (same depth bins):   {overlap_area_km2:,.2f}")
print(f"Unique Bathtub area (no common part): {unique_bath_total_km2:,.2f}")
print(f"Unique Process-based area:            {unique_proc_total_km2:,.2f}")

# (Optional) also print by depth bin
show_bins = False
if show_bins:
    for i in range(len(bins)-1):
        lo, hi = bins[i], bins[i+1]
        print(f"{lo:>4.1f}–{hi:<4.1f} m | Bath uniq: {unique_bath_bins[i]:7.3f} km² | "
              f"Proc uniq: {unique_proc_bins[i]:7.3f} km² | Overlap: {overlap_bins[i]:7.3f} km²")



=== Histogram-based area summary (km²) ===
Bathtub (orange) total area:         13,558.92
Process-based (blue) total area:      14,055.72
Overlapping area (same depth bins):   10,339.68
Unique Bathtub area (no common part): 3,219.24
Unique Process-based area:            3,716.04
